# 01. Abstract

`1` 희귀변이는 대립유전자(allele frequency)가 극히 낮기 때문에, 개별 변이의 효과를 추정하기 어렵다는 한계가 있음.

`2` 그래서 본 연구에서는 유전자(또는 특정 유전체 영역) 단위에서의 유전력(Heritability)을 먼저 추정한 뒤,

`3` 개별 변이에 대한 `effect size`를 `empirical Bayes approach`로 추정함.

`4` 본 논문에서는 `rare variant tests`에서 널리 사용되는 `Variance component model`을 사용하였으며,

`5` `gene/region level and variant level`에서의 효과 크기를 추출해 해석력을 높임.

`6` 또한, 희귀 질환 분석 등에서 흔히 보이는 `case-control`간 표본 불균형 문제를 보정하기 위해, `Firth bias correction` 알고리즘을 도입함.

`7` 그리고 `extensive simulations` and `analysis of UK Biobank`를 통해 높은 정확도와 연산 효율성을 입증함.

`8` RareEffect를 통해 구한 개별 변이 효과 크기($\hat{\beta}$)를 다유전자 위험 점수(Polygenic Risk Score, PRS) 계산에 활용하면 질환 예측력을 크게 높일 수 있으며, 최근 발표된 희귀 변이 PRS 전용 최신 모델들보다 훨씬 우수한 예측 성능($R^2$)

## Abstract summary

`1` 문제의식: 희귀 변이(Rare variants)는 빈도가 극히 낮아 단일 변이 단위의 효과 크기($\beta$) 추정이 불안정함.

`2` **제안 모델 (RareEffect)**: 유전자 단위의 유전율을 먼저 추정한 뒤, **경험적 베이즈(Empirical Bayes)** 방식으로 개별 변이 효과를 안정적으로 추정 (Firth 편향 보정으로 환자-대조군 불균형 해결).

`3` 성과 및 응용: UK Biobank 100개 형질 검증 결과 기존 기법 대비 우수한 예측력($R^2$)을 입증했으며, PRS에 반영 시 극단 고위험군 선별 정확도를 대폭 향상.

`4` 도출된 변이 효과 크기를 다유전자 점수(PRS)에 반영하여, **최근 개발된 희귀 변이 PRS 전용 모델들보다 우수한 예측 성능을 달성함**

---

# Introduction

## (1) 이전까지 희귀변이 연구에 대한 배경

`1` 대규모 시퀀싱 데이터를 활용할 수 있게 되어, 희귀변이에 대한 연구를 하는 것이 더 수월해 졌다는 이야기

>*"With the availability of extensive sequencing data from biobanks*[3]*, the study of rare variants has become more feasible than ever before."*

`2` 실제로 희귀 변이는 당뇨, 심혈관 질환, 암 등 수많은 복합 질환의 잠재적인 발병 원인으로 밝혀져 왔으며, 이러한 희귀 변이를 체계적을 규명하는 것은 복합 형질의 유전적 위험 구조를 명확히 이해하기 위한 핵심 과제임.

>"Rare variants are expected to have larger effect sizes than common variants, which can provide critical biological insights into disease etiology and mechanisms."

`3` 이전까지 희귀변이와 복잡한 질병들 사이의 연관성 연구는, 단일 변이 대신 여러 변이를 묶어서 검정하는 Burden, SKAT 그리고 SKAT-O 등이 활용되었음.

>"To identify associations between rare variants and complex traits, gene-based or region-based tests$^{10,11}$, including the burden test, the sequence kernel association test (SKAT)$^{12}$ and its adaptive optimized version (SKAT-O)$^{13}$, have been commonly used."

## (2) 새로운 방법들도 그닥..,.

`1` 희귀 변이의 영향을 명확하게 확인하기 위해서는, 변이들을 묶어서 검정한 후 p-value 값을 계산하는게 아니라 희귀변이별 effet size를 추정해야함.

> "To elucidate the effects of rare variants on complex diseases and traits and use them for risk prediction, in addition to calculating $P$ values for association tests, effect size estimation is required."

`2` 그래서 최근에 burden 처럼 변이들을 하나로 합치면 발생하는 변이 신호 상쇄 문제를 해결하기 위해, 실제로 새로운 방법론들이 개발됨.

>"Recently, several methods have been developed to estimate the effect sizes of rare variants, aiming to overcome the limitations of burden-test approaches$^{18–20}$

`3`  그러나 Burden의 한계를 극복하기 위해 제안되었던 이러한 기존 방법론들은 많은 연산을 필요로 하기 때문에 대규모 데이터 셋(`UK biobank`)에 적용하기가 힘듬

> "However, these methods often require excessive computational resources, making their application to large-scale datasets impractical."

## (3) 해결책

`1` 그래서 `Empirical Bayes` 기반의 `RareEffect`라는 개별 변이 효과를 추정하는 방법을 소개한다.

> "To address these challenges, we introduce RareEffect, an empirical Bayes approach designed to estimate the effect sizes of individual rare variants while simultaneously estimating gene-level heritability."

`2` `Vaiance component Model`기반의 SKAT와 유사함. 단, 유전자 단위와 변이 단위로 `effect size`를 추정한다.

> "Unlike the burden test, RareEffect can accommodate variants with opposing effect directions, allowing for a more accurate reflection of variant-level effect sizes."

`3` 또한 유전자 단위 분산 성분을 추정할 때 발생하는 막대한 연산 비용을 획기적으로 줄이기 위해, 본 연구진은 유전적 혈연 관계 행렬(GRM)의 낮은 랭크(low-rank) 성질을 활용하는 고속 선형 혼합 모델인 'FaST-LMM' 알고리즘을 도입함.

> To reduce the computational burden of variance components estimation, we implemented factored spectrally transformed linear mixed models (FaST-LMM)$^{23}$, which leverage the low rank of the genetic relatedness matrix."

`4` 이에 더해, `sparsity` 문제를 해결하기 위해 `SAIGE-GENE+`의 `collapsing ultrarare variants`을 도입함. 그래서 초희귀 변이들의 `effects` 추정에 대한 검정력을 높임.

> "We also adopted the strategy of collapsing ultrarare variants, as used in SAIGE-GENE+, to reduce the sparsity of the genotype matrix and improve the power to estimate the effects of ultrarare variants$^{16}$."

`5` `Firth bias reduction`: effect size를 stably하게 추정하기 위한 방법... 일단 이 정도로만...

> For binary traits, we additionally applied a fast implementation of the Firth bias reduction method to stably estimate effect sizes.

## (4) 주요 결과

`1` 개별 희귀 변이의 효과 크기($\beta$)를 추정할 때, 기존의 단일 변이 선형 회귀나 릿지(Ridge) 회귀 대비 평균 제곱근 오차(RMSE)를 크게 낮추어 훨씬 더 정밀한 추정 정확도를 달성

> "We also show that the proposed approach outperforms linear regression and ridge regression in terms of root mean squared error (RMSE) for estimating individual variant-level effect sizes."

`2`  실제 UK Biobank 100개 복합 형질의 엑솜 데이터를 분석한 결과, 엑손 내 희귀 변이들이 기존 흔한 변이(GWAS)가 설명하지 못했던 **phenotypic variability** 설명함을 입증했습니다 (단, 기여도는 질환/형질 종류에 따라 차이가 있음).

> From the UK Biobank (UKB) whole-exome sequencing (WES) data analysis of 100 traits, we demonstrate that exonic rare variants can explain additional phenotypic variability, but the degree differs by phenotype.

`3` AlphaMissense나 PrimateAI-3D 같은 AI 병원성 예측 점수, 혹은 DeepRVAT 같은 유전자 손상 점수를 활용하는 최신 방법론들보다, **실제 임상 표현형의 변이를 훨씬 더 정확하게 설명**해냄.

> We also demonstrate that our approach explains phenotypic variability more effectively than recently developed methods based on predicted variant functional impact or gene impairment scores, such as AlphaMissense, PrimateAI-3D and DeepRVAT.

`4` 뭐, 자기들이 제안한 method가 `risk prediction`과 `heritability estimation` 추정에서  `strong evidence`를 확보했다. 뭐 그런 이야기임. 

> *"These findings provide strong evidence for the practical utility of our method in leveraging rare variant data for risk prediction and heritability estimation."*